# 01 · The forward process, schedules & the "nice property"

The forward process adds a little Gaussian noise at each step $t$:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\big(x_t;\ \sqrt{1-\beta_t}\,x_{t-1},\ \beta_t I\big)$$

The sequence $\beta_1,\dots,\beta_T$ is the **noise schedule**. Define
$\alpha_t = 1-\beta_t$ and the cumulative product
$\bar\alpha_t=\prod_{s\le t}\alpha_s$ — *how much of the original signal survives*
to step $t$.

> ### 📝 How this notebook works
> Cells marked **`# TODO`** are yours to write. Each one is followed by a
> **self-check** cell — run it and it will tell you if your implementation is
> correct (it compares against the reference and asserts the key properties).
>
> **Stuck?** The answer key is `solutions/notebooks/` (with plots), and the
> reference implementation lives in the `nanodiffusion/` package. Peeking is
> allowed — but try first.

In [ ]:
import torch
import matplotlib.pyplot as plt

from nanodiffusion.utils import set_seed, scatter_2d
from nanodiffusion.data import toy2d
# reference implementations, used ONLY by the self-check cells:
from nanodiffusion.schedules import NoiseSchedule, linear_beta_schedule, cosine_beta_schedule
from nanodiffusion.forward import add_noise as reference_add_noise

set_seed(0)
T = 200
data = toy2d("swiss_roll", 8000)
print("data:", tuple(data.shape))

## TODO 1 — derive $\bar\alpha_t$ from the betas

Given a tensor of `betas` of length $T$, compute:

- `alphas`      $= 1 - \beta_t$
- `alpha_bars`  $= \prod_{s \le t} \alpha_s$   ← this is a **cumulative product**

*Hint:* `torch.cumprod(x, dim=0)`.

In [ ]:
def my_alpha_bars(betas: torch.Tensor) -> torch.Tensor:
    '''Return alpha_bar_t = prod_{s<=t} (1 - beta_s), same length as betas.'''
    # TODO: implement (2 lines)
    raise NotImplementedError

In [ ]:
# ---- self-check 1 ----
betas = linear_beta_schedule(T)
mine = my_alpha_bars(betas)
ref = NoiseSchedule(betas).alpha_bars
assert mine.shape == ref.shape, f"shape {mine.shape} != {ref.shape}"
assert torch.allclose(mine, ref, atol=1e-6), "values don't match the reference"
assert torch.all(mine[:-1] >= mine[1:]), "alpha_bar must be non-increasing"
print("✅ TODO 1 correct — signal decays from"
      f" {mine[0]:.3f} to {mine[-1]:.4f}")

## The two schedules

DDPM used a **linear** $\beta$ schedule; "improved DDPM" introduced a **cosine**
one that destroys signal more gently early on. Compare them with your function.

In [ ]:
lin_ab = my_alpha_bars(linear_beta_schedule(T))
cos_ab = my_alpha_bars(cosine_beta_schedule(T))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(linear_beta_schedule(T), label="linear"); a1.plot(cosine_beta_schedule(T), label="cosine")
a1.set_title(r"$\beta_t$"); a1.set_xlabel("t"); a1.legend()
a2.plot(lin_ab, label="linear"); a2.plot(cos_ab, label="cosine")
a2.set_title(r"$\bar\alpha_t$ (signal surviving)"); a2.set_xlabel("t"); a2.legend()
plt.tight_layout(); plt.show()

## TODO 2 — the "nice property"

This is *the* equation that makes diffusion trainable. Because sums of Gaussians
are Gaussian, we can jump from $x_0$ straight to $x_t$ in **one shot** — no need
to simulate $t$ little steps:

$$\boxed{\,x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \varepsilon\sim\mathcal N(0,I)\,}$$

Implement it. Note the shape juggling: `x0` is `(B, 2)` and `t` is `(B,)`, so
after indexing `alpha_bars[t]` you must reshape to `(B, 1)` to broadcast.

In [ ]:
def my_add_noise(x0: torch.Tensor, t: torch.Tensor, alpha_bars: torch.Tensor,
                 noise: torch.Tensor | None = None):
    '''Noise x0 to timestep t in one shot.

    Args:
        x0:         (B, 2) clean data
        t:          (B,) long tensor of timesteps
        alpha_bars: (T,) the cumulative products from TODO 1
        noise:      optional (B, 2) eps; sample from N(0, I) if None
    Returns:
        (x_t, noise)
    '''
    if noise is None:
        noise = torch.randn_like(x0)
    # TODO:
    #   1. gather alpha_bar_t for each item:      ab = alpha_bars[t]
    #   2. reshape it to (B, 1) so it broadcasts: ab = ab.reshape(-1, 1)
    #   3. return  sqrt(ab) * x0 + sqrt(1 - ab) * noise,  and the noise
    raise NotImplementedError

In [ ]:
# ---- self-check 2 ----
schedule = NoiseSchedule.make("cosine", T)
ab = my_alpha_bars(cosine_beta_schedule(T))
t = torch.randint(0, T, (data.shape[0],))
eps = torch.randn_like(data)                      # fixed noise, so we can compare

x_mine, _ = my_add_noise(data, t, ab, noise=eps)
x_ref, _ = reference_add_noise(data, t, schedule, noise=eps)
assert x_mine.shape == data.shape, f"shape {x_mine.shape} != {data.shape}"
assert torch.allclose(x_mine, x_ref, atol=1e-5), "doesn't match the reference"

# at t = T-1 the data should be ~pure unit-variance noise
x_T, _ = my_add_noise(data, torch.full((data.shape[0],), T - 1), ab)
print(f"alpha_bar_T = {ab[-1]:.4f}  (want ~0)")
print(f"std(x_T)    = {x_T.std().item():.4f}  (want ~1)")
assert abs(x_T.std().item() - 1.0) < 0.15
print("✅ TODO 2 correct — the forward process works")

## See it work

Use **your** `my_add_noise` to watch the swiss roll dissolve.

In [ ]:
ts = [0, 10, 25, 50, 100, 199]
fig, axes = plt.subplots(1, len(ts), figsize=(2.6 * len(ts), 2.6))
for ax, ti in zip(axes, ts):
    x_t, _ = my_add_noise(data, torch.full((data.shape[0],), ti), ab)
    scatter_2d(ax, x_t, f"t = {ti}")
plt.suptitle("Your forward process: swiss roll -> noise")
plt.tight_layout(); plt.show()

✅ **Done.** You implemented the forward process. Compare yours to
`nanodiffusion/schedules.py` and `nanodiffusion/forward.py`.

Next: notebook 02 — teach a network to *undo* this.